# EDA — DVF (Demandes de Valeurs Foncières)
Urban Data Explorer · data.gouv.fr / DGFiP

> Dataset des transactions immobilières enregistrées en France.  
> 1 ligne = 1 bien vendu (appartement, maison, terrain, dépendance…).  
> Source : Direction générale des Finances publiques (DGFiP).


## 0. Imports & configuration

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 130
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11


## 1. Chargement

In [11]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

FILE = '../brute/Nouveau dossier/dvf.csv'
OUT  = '../brute/Nouveau dossier/dvf2.parquet'

cols = pd.read_csv(FILE, nrows=0, sep=',').columns.tolist()

# Schéma fixe : toutes les colonnes en string
schema = pa.schema([(c, pa.string()) for c in cols])
writer = pq.ParquetWriter(OUT, schema)

for chunk in pd.read_csv(FILE, sep=',', dtype=str, chunksize=100_000):
    for c in cols:
        if c not in chunk.columns:
            chunk[c] = None
    chunk = chunk[cols].astype(str).replace('nan', None)
    table = pa.Table.from_pandas(chunk, schema=schema, preserve_index=False)
    writer.write_table(table)

writer.close()
print('Converti :', OUT)

Converti : ../brute/Nouveau dossier/dvf2.parquet


## 2. Infos générales

In [ ]:
df.info()


In [ ]:
# Parsing date
df['date_mutation'] = pd.to_datetime(df['date_mutation'], errors='coerce')
df['annee']  = df['date_mutation'].dt.year
df['mois']   = df['date_mutation'].dt.month
df['jour_semaine'] = df['date_mutation'].dt.day_name()

print(f'Période couverte : {df["date_mutation"].min().date()} → {df["date_mutation"].max().date()}')
print(f'Années présentes : {sorted(df["annee"].dropna().unique().astype(int))}')
print()
print('Répartition par nature de mutation :')
print(df['nature_mutation'].value_counts())
print()
print('Répartition par type local :')
print(df['type_local'].value_counts())


### Dictionnaire des colonnes clés

| Colonne | Description |
|---|---|
| `id_mutation` | Identifiant unique de la transaction |
| `date_mutation` | Date de la vente |
| `nature_mutation` | Vente, adjudication, expropriation… |
| `valeur_fonciere` | Prix de vente (€) |
| `type_local` | Appartement, Maison, Local industriel, Dépendance |
| `surface_reelle_bati` | Surface bâtie (m²) |
| `nombre_pieces_principales` | Nombre de pièces |
| `surface_terrain` | Surface du terrain (m²) |
| `code_departement` | Code département |
| `code_commune` | Code INSEE commune |
| `longitude` / `latitude` | Coordonnées GPS |
| `lot1..5_surface_carrez` | Surfaces loi Carrez par lot |


## 3. Valeurs manquantes

In [ ]:
missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing = missing[missing > 0]

fig, ax = plt.subplots(figsize=(10, max(4, len(missing) * 0.38)))
colors = ['#D85A30' if v > 70 else '#BA7517' if v > 30 else '#1D9E75' for v in missing.values]
ax.barh(missing.index, missing.values, color=colors)
ax.set_xlabel('% manquant')
ax.set_title('Valeurs manquantes par colonne (rouge > 70 %, orange > 30 %)')
ax.axvline(70, color='#D85A30', linestyle='--', linewidth=0.8, alpha=0.5)
ax.axvline(30, color='#BA7517', linestyle='--', linewidth=0.8, alpha=0.5)
plt.tight_layout()
plt.show()

display(missing.rename('% manquant').to_frame().style.format('{:.1f}%'))


In [ ]:
# Explication structurelle des NaN
print("Colonnes à NaN structurels :")
print("  lot2..5_surface_carrez  : absent si la transaction ne concerne qu'un seul lot")
print("  nature_culture_speciale : absent pour les biens bâtis")
print("  ancien_code/nom_commune : absent si commune non fusionnée")
print("  surface_terrain         : absent pour les appartements")
print("  surface_reelle_bati     : absent pour les terrains nus")
print()
print(f"Taux de NaN valeur_fonciere  : {df['valeur_fonciere'].isna().mean()*100:.1f} %")
print(f"Taux de NaN surface_reelle   : {df['surface_reelle_bati'].isna().mean()*100:.1f} %")
print(f"Taux de NaN latitude         : {df['latitude'].isna().mean()*100:.1f} %")


## 4. Doublons

In [ ]:
n_dup = df.duplicated().sum()
print(f'Doublons stricts : {n_dup} ({n_dup/len(df)*100:.2f} %)')

# Dans DVF, une même id_mutation peut avoir plusieurs lignes (multi-lots / multi-biens)
print()
print('Lignes par id_mutation (distribution) :')
n_par_mut = df.groupby('id_mutation').size()
print(n_par_mut.value_counts().head(10).rename('nb mutations'))
print(f'=> {(n_par_mut > 1).sum():,} mutations multi-lignes sur {n_par_mut.nunique():,} mutations uniques')


## 5. Valeur foncière (prix de vente)

In [ ]:
# Une mutation multi-lignes = même valeur foncière sur chaque ligne → dédupliquer
df_mut = df.drop_duplicates(subset='id_mutation')[['id_mutation','date_mutation','annee',
          'nature_mutation','valeur_fonciere','type_local','code_departement']].copy()

vf = df_mut['valeur_fonciere'].dropna()
print(f'Transactions avec prix : {len(vf):,}')
print(f'Médiane  : {vf.median():,.0f} €')
print(f'Moyenne  : {vf.mean():,.0f} €')
print(f'Min      : {vf.min():,.0f} €')
print(f'Max      : {vf.max():,.0f} €')
print(f'< 1 000 € (probables erreurs/lots) : {(vf < 1000).sum():,}')


In [ ]:
# Filtrage valeurs aberrantes pour la visualisation
vf_plot = vf[(vf >= 1000) & (vf <= vf.quantile(0.99))]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].hist(vf_plot / 1000, bins=60, color='#378ADD', alpha=0.85, edgecolor='white', linewidth=0.3)
axes[0].axvline(vf_plot.median() / 1000, color='#D85A30', linestyle='--', linewidth=1.5,
                label=f'Médiane : {vf_plot.median()/1000:.0f} k€')
axes[0].set_xlabel('Valeur foncière (k€)')
axes[0].set_ylabel('Nombre de transactions')
axes[0].set_title('Distribution des prix (hors top 1 %)', fontweight='bold')
axes[0].legend()

# Log scale
axes[1].hist(np.log10(vf[vf > 0]), bins=60, color='#1D9E75', alpha=0.85, edgecolor='white', linewidth=0.3)
axes[1].set_xlabel('log10(valeur foncière)')
axes[1].set_ylabel('Nombre de transactions')
axes[1].set_title('Distribution des prix (échelle log10)', fontweight='bold')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{10**x/1000:.0f} k€'))

plt.tight_layout()
plt.show()


In [ ]:
# Prix médian par type de bien
prix_type = (df_mut[df_mut['valeur_fonciere'] >= 1000]
             .groupby('type_local')['valeur_fonciere']
             .describe(percentiles=[.25,.5,.75]))
display(prix_type.style.format('{:,.0f}'))


## 6. Prix au m² — biens bâtis

In [ ]:
# Focus appartements et maisons avec surface connue
mask = (
    df['type_local'].isin(['Appartement','Maison']) &
    df['surface_reelle_bati'].notna() &
    df['valeur_fonciere'].notna() &
    (df['surface_reelle_bati'] > 10) &
    (df['valeur_fonciere'] >= 1000)
)
df_bati = df[mask].copy()

# Agréger par mutation (valeur foncière unique par mutation)
df_bati_mut = df_bati.groupby('id_mutation').agg(
    valeur_fonciere=('valeur_fonciere','first'),
    surface_reelle_bati=('surface_reelle_bati','sum'),
    type_local=('type_local','first'),
    code_departement=('code_departement','first'),
    annee=('annee','first'),
    nombre_pieces_principales=('nombre_pieces_principales','first'),
    latitude=('latitude','first'),
    longitude=('longitude','first'),
).reset_index()

df_bati_mut['prix_m2'] = df_bati_mut['valeur_fonciere'] / df_bati_mut['surface_reelle_bati']

# Filtrage outliers prix/m²
q01 = df_bati_mut['prix_m2'].quantile(0.01)
q99 = df_bati_mut['prix_m2'].quantile(0.99)
df_bati_mut = df_bati_mut[(df_bati_mut['prix_m2'] >= q01) & (df_bati_mut['prix_m2'] <= q99)]

print(f'Biens bâtis avec prix/m² valide : {len(df_bati_mut):,}')
print()
print(df_bati_mut.groupby('type_local')['prix_m2'].describe(percentiles=[.25,.5,.75]).round(0).to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for ax, t in zip(axes, ['Appartement','Maison']):
    data = df_bati_mut[df_bati_mut['type_local'] == t]['prix_m2']
    ax.hist(data, bins=60, color='#378ADD' if t=='Appartement' else '#1D9E75',
            alpha=0.85, edgecolor='white', linewidth=0.3)
    ax.axvline(data.median(), color='#D85A30', linestyle='--', linewidth=1.5,
               label=f'Médiane : {data.median():,.0f} €/m²')
    ax.set_xlabel('Prix au m² (€)')
    ax.set_ylabel('Nombre de transactions')
    ax.set_title(f'Prix au m² — {t}', fontweight='bold')
    ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
# Prix/m² par département (top 20 en volume)
top_dep = df_bati_mut.groupby('code_departement').size().nlargest(20).index
df_dep = df_bati_mut[df_bati_mut['code_departement'].isin(top_dep)]

fig, ax = plt.subplots(figsize=(13, 7))
order = (df_dep.groupby('code_departement')['prix_m2']
               .median().sort_values(ascending=False).index)
sns.boxplot(data=df_dep, x='code_departement', y='prix_m2',
            order=order, palette='muted', ax=ax, fliersize=2)
ax.set_xlabel('Département')
ax.set_ylabel('Prix au m² (€)')
ax.set_title('Distribution du prix au m² — top 20 départements (vol. transactions)', fontweight='bold')
plt.tight_layout()
plt.show()


## 7. Évolution temporelle

In [ ]:
# Volume mensuel de transactions
vol_mois = (df.drop_duplicates('id_mutation')
              .groupby(df['date_mutation'].dt.to_period('M'))
              .size()
              .reset_index(name='n_transactions'))
vol_mois['date'] = vol_mois['date_mutation'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(vol_mois['date'], vol_mois['n_transactions'], color='#378ADD', linewidth=1.2)
ax.fill_between(vol_mois['date'], vol_mois['n_transactions'], alpha=0.2, color='#378ADD')
ax.set_xlabel('Date')
ax.set_ylabel('Nb transactions')
ax.set_title('Volume mensuel de transactions immobilières', fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Prix médian par mois
prix_mois = (df_bati_mut.dropna(subset=['annee'])
             .assign(periode=pd.to_datetime(df_bati_mut['annee'].astype(int).astype(str) + '-01-01'))
             .groupby('annee')['prix_m2']
             .median()
             .reset_index(name='prix_m2_median'))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(prix_mois['annee'], prix_mois['prix_m2_median'],
        marker='o', color='#D85A30', linewidth=1.5)
ax.set_xlabel('Année')
ax.set_ylabel('Prix médian au m² (€)')
ax.set_title('Évolution du prix médian au m² (biens bâtis)', fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%,.0f €'))
plt.tight_layout()
plt.show()


## 8. Surface et nombre de pièces

In [ ]:
surf = df_bati_mut['surface_reelle_bati']
pieces = df_bati_mut['nombre_pieces_principales'].dropna()

print(f'Surface bâtie — médiane : {surf.median():.0f} m²  |  max : {surf.max():.0f} m²')
print(f'Pièces — médiane : {pieces.median():.0f}  |  mode : {pieces.mode()[0]:.0f}')


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Surface
axes[0].hist(surf[surf <= surf.quantile(0.99)], bins=50,
             color='#7B52A9', alpha=0.85, edgecolor='white', linewidth=0.3)
axes[0].set_xlabel('Surface (m²)')
axes[0].set_ylabel('Nb biens')
axes[0].set_title('Distribution surface bâtie', fontweight='bold')

# Pièces
p_counts = pieces[pieces.between(1, 10)].value_counts().sort_index()
axes[1].bar(p_counts.index, p_counts.values, color='#1D9E75', alpha=0.85, edgecolor='white')
axes[1].set_xlabel('Nombre de pièces')
axes[1].set_ylabel('Nb biens')
axes[1].set_title('Distribution du nombre de pièces', fontweight='bold')

# Surface vs prix/m²
sub = df_bati_mut[df_bati_mut['surface_reelle_bati'] <= df_bati_mut['surface_reelle_bati'].quantile(0.99)]
axes[2].scatter(sub['surface_reelle_bati'], sub['prix_m2'],
                alpha=0.2, s=5, color='#378ADD')
axes[2].set_xlabel('Surface bâtie (m²)')
axes[2].set_ylabel('Prix au m² (€)')
axes[2].set_title('Surface vs prix au m²', fontweight='bold')

plt.tight_layout()
plt.show()


## 9. Analyse géographique

In [ ]:
# Répartition par département
dep_vol = (df.drop_duplicates('id_mutation')
             .groupby('code_departement')
             .size()
             .sort_values(ascending=False)
             .head(25)
             .reset_index(name='n_transactions'))

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.bar(dep_vol['code_departement'], dep_vol['n_transactions'],
              color='#378ADD', alpha=0.85, edgecolor='white')
ax.set_xlabel('Département')
ax.set_ylabel('Nb transactions')
ax.set_title('Volume de transactions — top 25 départements', fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Scatter géographique (si latitude/longitude disponibles)
geo = df_bati_mut.dropna(subset=['latitude','longitude'])
geo = geo[(geo['latitude'].between(41, 51.5)) & (geo['longitude'].between(-5, 10))]

if len(geo) > 0:
    fig, ax = plt.subplots(figsize=(10, 10))
    sc = ax.scatter(geo['longitude'], geo['latitude'],
                    c=np.log10(geo['prix_m2'].clip(lower=100)),
                    cmap='RdYlGn_r', s=1, alpha=0.3)
    cbar = fig.colorbar(sc, ax=ax, shrink=0.6)
    cbar.set_label('log10(prix/m²)')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title('Carte des transactions — couleur = prix/m²', fontweight='bold')
    ax.set_aspect('equal')
    plt.tight_layout()
    plt.show()
    print(f'Points affichés : {len(geo):,}')
else:
    print('Coordonnées géographiques non disponibles.')


## 10. Lots et copropriétés

In [ ]:
# Nombre de lots par transaction
print('Distribution nombre_lots :')
print(df['nombre_lots'].value_counts().head(10))
print()

# Surface Carrez lot1
sc1 = df['lot1_surface_carrez'].dropna()
print(f'lot1_surface_carrez — n={len(sc1):,}  médiane={sc1.median():.1f} m²  max={sc1.max():.1f} m²')


In [ ]:
# Proportion des types de local
fig, ax = plt.subplots(figsize=(8, 8))
type_counts = df['type_local'].value_counts()
wedge_colors = ['#378ADD','#1D9E75','#D85A30','#BA7517','#7B52A9']
ax.pie(type_counts.values, labels=type_counts.index,
       autopct='%1.1f%%', colors=wedge_colors[:len(type_counts)],
       startangle=90, pctdistance=0.82)
ax.set_title('Répartition par type de local', fontweight='bold', pad=20)
plt.tight_layout()
plt.show()


## 11. Table silver — biens bâtis prêts pour le pipeline

In [ ]:
cols_silver = ['id_mutation','date_mutation','annee',
              'nature_mutation','type_local',
              'valeur_fonciere','surface_reelle_bati','prix_m2',
              'nombre_pieces_principales',
              'code_departement','code_commune','nom_commune',
              'latitude','longitude']

df_silver = df_bati_mut[[c for c in cols_silver if c in df_bati_mut.columns]].copy()
df_silver = df_silver.rename(columns={
    'valeur_fonciere'         : 'prix_vente_eur',
    'surface_reelle_bati'     : 'surface_m2',
    'nombre_pieces_principales': 'nb_pieces',
})

print(f'Table silver : {df_silver.shape[0]:,} transactions × {df_silver.shape[1]} colonnes')
print(f'Prix/m² : médiane {df_silver["prix_m2"].median():,.0f} €/m²')
display(df_silver.head(5))


## 12. Résumé EDA

In [ ]:
print("=== RÉSUMÉ EDA — DVF (Demandes de Valeurs Foncières) ===\n")
print(f'Lignes totales              : {len(df):,}')
print(f'Mutations uniques           : {df["id_mutation"].nunique():,}')
print(f'Colonnes                    : {len(df.columns)}')
print(f'Période couverte            : {df["date_mutation"].min().date()} → {df["date_mutation"].max().date()}')
print()
print('── Biens ──')
print(df['type_local'].value_counts().to_string())
print()
print('── Prix ──')
vf_ventes = df.drop_duplicates('id_mutation')['valeur_fonciere'].dropna()
vf_ventes = vf_ventes[vf_ventes >= 1000]
print(f'  Médiane prix de vente     : {vf_ventes.median():,.0f} €')
if 'prix_m2' in df_bati_mut.columns:
    print(f'  Médiane prix/m² bâti      : {df_bati_mut["prix_m2"].median():,.0f} €/m²')
print()
print('── Qualité données ──')
print(f'  NaN valeur_fonciere       : {df["valeur_fonciere"].isna().mean()*100:.1f} %')
print(f'  NaN surface_reelle_bati   : {df["surface_reelle_bati"].isna().mean()*100:.1f} %')
print(f'  NaN latitude              : {df["latitude"].isna().mean()*100:.1f} %')
print(f'  NaN structurels           : lots multiples, terrains sans surface bâtie, etc.')
